In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report,
    average_precision_score
)

In [2]:
df= pd.read_csv('../../data/social_media_ad_optimization_raw.csv') 
df.head()

,user_id,age,gender,location,interests,ad_id,ad_category,ad_platform,ad_type,impressions,clicks,conversion,time_spent_on_ad,day_of_week,device_type,engagement_score
0,U0001,58,M,USA,Food,A0001,Sportswear,Facebook,Image,3,0,0,3.38,Friday,Mobile,0.02
1,U0002,55,F,USA,Tech,A0002,Electronics,Facebook,Image,9,9,1,6.77,Saturday,Tablet,0.93
2,U0003,52,F,UK,Gaming,A0003,Luggage,Instagram,Image,13,12,1,13.26,Wednesday,Mobile,0.93
3,U0004,31,F,USA,Tech,A0004,Gadgets,Facebook,Video,14,5,0,24.41,Saturday,Desktop,0.28
4,U0005,52,M,India,Tech,A0005,Luggage,Instagram,Carousel,10,5,0,21.43,Monday,Tablet,0.35


In [3]:
# engagement_score is excluded: it correlates ~0.81 with conversion and is
# effectively derived from the outcome, which would leak the target
needed = [
    'day_of_week','device_type','location',
    'ad_platform','ad_type','ad_category',
    'time_spent_on_ad',
    'conversion'
]
df_model = df[needed].dropna().copy()

In [4]:
X = df_model.drop('conversion', axis=1)
y = df_model['conversion']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

In [5]:
categorical_cols = ['day_of_week','device_type','location','ad_platform','ad_type','ad_category']
numeric_cols     = ['time_spent_on_ad']

preprocess = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols),
    ('num', StandardScaler(), numeric_cols)
])

In [6]:
rf = RandomForestClassifier(
    n_estimators=600,
    max_depth=14,
    min_samples_leaf=2,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)


In [7]:
model = Pipeline([
    ('prep', preprocess),
    ('rf', rf)
])

In [8]:
model.fit(X_train, y_train)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['day_of_week', 'device_type',
                                                   'location', 'ad_platform',
                                                   'ad_type', 'ad_category']),
                                                 ('num', StandardScaler(),
                                                  ['time_spent_on_ad'])])),
                ('rf',
                 RandomForestClassifier(class_weight='balanced', max_depth=14,
                                        min_samples_leaf=2, n_estimators=600,
                                        n_jobs=-1, random_state=42))])

In [9]:
y_proba = model.predict_proba(X_test)[:, 1]
y_pred  = (y_proba >= 0.5).astype(int)

print("Test Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Test ROC-AUC:", round(roc_auc_score(y_test, y_proba), 3))
print("Test PR-AUC (Avg Precision):", round(average_precision_score(y_test, y_proba), 3))
print(classification_report(y_test, y_pred))

Test Accuracy: 0.6
Test ROC-AUC: 0.604
Test PR-AUC (Avg Precision): 0.556
              precision    recall  f1-score   support

           0       0.63      0.58      0.61        53
           1       0.57      0.62      0.59        47

    accuracy                           0.60       100
   macro avg       0.60      0.60      0.60       100
weighted avg       0.60      0.60      0.60       100



In [10]:
cv_acc = cross_val_score(model, X, y, cv=5, scoring='accuracy').mean()
cv_auc = cross_val_score(model, X, y, cv=5, scoring='roc_auc').mean()
print("CV Accuracy (5-fold mean):", round(cv_acc, 3))
print("CV ROC-AUC (5-fold mean):", round(cv_auc, 3))

CV Accuracy (5-fold mean): 0.562
CV ROC-AUC (5-fold mean): 0.603


In [11]:
importances = model.named_steps['rf'].feature_importances_
feat_names  = model.named_steps['prep'].get_feature_names_out()
fi = (pd.DataFrame({'feature': feat_names, 'importance': importances})
        .sort_values('importance', ascending=False))
print("\nTop features:\n", fi.head(10))




Top features:
                              feature  importance
21             num__time_spent_on_ad    0.321753
13        cat__ad_platform_Instagram    0.045704
6            cat__device_type_Mobile    0.043928
15                cat__ad_type_Video    0.041179
7            cat__device_type_Tablet    0.039413
10               cat__location_India    0.037082
14                cat__ad_type_Image    0.034824
12                 cat__location_USA    0.033280
17  cat__ad_category_Food & Beverage    0.033145
1          cat__day_of_week_Saturday    0.033039


In [12]:
train_df = X_train.copy()
train_df['conversion'] = y_train.values

def mode_or_global(s, global_mode):
    return s.mode().iloc[0] if not s.mode().empty else global_mode

global_modes = {
    'ad_platform': train_df['ad_platform'].mode().iloc[0],
    'ad_type'    : train_df['ad_type'].mode().iloc[0],
    'ad_category': train_df['ad_category'].mode().iloc[0],
}


In [13]:
medians = {
    'time_spent_on_ad': float(train_df['time_spent_on_ad'].median()),
}

In [14]:
unique_days      = X_train['day_of_week'].unique()
unique_devices   = X_train['device_type'].unique()
unique_countries = X_train['location'].unique()

rows = []
for country in unique_countries:
    # country-specific defaults (fallback to global)
    cdf = train_df[train_df['location'] == country]
    plat = mode_or_global(cdf['ad_platform'], global_modes['ad_platform'])
    atype = mode_or_global(cdf['ad_type'], global_modes['ad_type'])
    acat = mode_or_global(cdf['ad_category'], global_modes['ad_category'])

    for day in unique_days:
        for device in unique_devices:
            sample = pd.DataFrame({
                'day_of_week':      [day],
                'device_type':      [device],
                'location':         [country],
                'ad_platform':      [plat],
                'ad_type':          [atype],
                'ad_category':      [acat],
                'time_spent_on_ad': [medians['time_spent_on_ad']],
            })
            p = model.predict_proba(sample)[:, 1][0]
            rows.append([country, day, device, plat, atype, acat,
                         medians['time_spent_on_ad'], p])

In [15]:
grid = pd.DataFrame(rows, columns=[
    'country','day','device','assumed_platform','assumed_ad_type','assumed_category',
    'assumed_time_spent','conversion_probability'
])

In [16]:
top3_per_country = (grid
    .sort_values(['country','conversion_probability'], ascending=[True, False])
    .groupby('country')
    .head(3)
    .reset_index(drop=True))

print("\nTop day–device combinations per country (holding other features at realistic defaults):")
print(top3_per_country[['country','day','device','conversion_probability']])


Top day–device combinations per country (holding other features at realistic defaults):
      country        day   device  conversion_probability
0   Australia   Thursday  Desktop                0.519238
1   Australia    Tuesday  Desktop                0.502461
2   Australia   Thursday   Mobile                0.491948
3      Canada     Sunday  Desktop                0.729220
4      Canada     Sunday   Mobile                0.727668
5      Canada   Thursday   Mobile                0.707011
6     Germany   Thursday   Mobile                0.727861
7     Germany  Wednesday   Mobile                0.721712
8     Germany     Friday   Mobile                0.703948
9       India    Tuesday  Desktop                0.638398
10      India    Tuesday   Mobile                0.576418
11      India   Thursday  Desktop                0.567102
12         UK   Saturday  Desktop                0.505694
13         UK     Sunday  Desktop                0.504315
14         UK   Saturday   Tablet        

In [17]:
# Check training vs. test performance
y_train_pred = model.predict(X_train)
train_acc = accuracy_score(y_train, y_train_pred)
train_auc = roc_auc_score(y_train, model.predict_proba(X_train)[:,1])
print("Train accuracy:", round(train_acc, 3), "| Train ROC-AUC:", round(train_auc, 3))

Train accuracy: 0.948 | Train ROC-AUC: 0.99
